## Utilities

In [2]:
import sys
import pandas as pd

sys.path.append('..')
from utils.db_utils import write_table, read_table


## Reading target tables

### Supply: Employed Graduates by Sector

In [15]:
df_supply = read_table("""
    select * from sc_bronze.dosm_emp_graduates_sector
""")

df_supply.head(10)

,year,sector,total_emp_graduate
0,2016,Agriculture,42500.0
1,2016,Mining and Quarrying,48600.0
2,2016,Manufacturing,424800.0
3,2016,Construction,202500.0
4,2016,Services,2758500.0
5,2017,Agriculture,46200.0
6,2017,Mining and Quarrying,39900.0
7,2017,Manufacturing,486800.0
8,2017,Construction,219800.0
9,2017,Services,2887300.0


### Demand: Job Availability by Sector

In [16]:
df_demand = read_table("""
    select 
    date,
    sector,
    sum(job_available) as job_available,
    sum(job_filled) as job_filled,
    sum(job_vacancy) as job_vacancy
    from sc_bronze.dosm_jobdemand
    group by date, sector
    order by date, sector
""")

df_demand.head(10)

,date,sector,job_available,job_filled,job_vacancy
0,2018-01-01,Agriculture,474600.0,445100.0,29500.0
1,2018-01-01,Construction,1315100.0,1293900.0,21200.0
2,2018-01-01,Manufacturing,2216100.0,2107900.0,108200.0
3,2018-01-01,Mining and Quarrying,81900.0,81500.0,400.0
4,2018-01-01,Services,4380100.0,4337000.0,43000.0
5,2018-04-01,Agriculture,486600.0,457100.0,29500.0
6,2018-04-01,Construction,1312400.0,1290400.0,21800.0
7,2018-04-01,Manufacturing,2218800.0,2110600.0,108000.0
8,2018-04-01,Mining and Quarrying,81900.0,81600.0,400.0
9,2018-04-01,Services,4374500.0,4334100.0,40500.0


## Data Processing

In [5]:
def align_and_merge(df_supply, df_demand):
    df_supply['date'] = pd.to_datetime(df_supply['year'].astype(str) + '-01-01')
    
    # Upsample from Annual to Quarterly
    supply_q = df_supply.set_index('date').groupby('sector')['total_emp_graduate'] \
                .resample('QS').interpolate(method='linear').reset_index()
    
    # Merge with Demand Data
    df_demand['date'] = pd.to_datetime(df_demand['date'])
    df = pd.merge(df_demand, supply_q, on=['date', 'sector'], how='inner')
    
    return df

## Feature Engineering

In [6]:
def engineer_features(df):
    # Total available jobs and graduates per quarter
    
    df['total_q_jobs'] = df.groupby('date')['job_available'].transform('sum')
    df['total_q_grads'] = df.groupby('date')['total_emp_graduate'].transform('sum')
    
    # Calculate Percentage Shares
    df['demand_pct'] = (df['job_available'] / df['total_q_jobs']) * 100
    df['supply_pct'] = (df['total_emp_graduate'] / df['total_q_grads']) * 100
    
    # Mismatch Gap
    df['gap_pct'] = df['demand_pct'] - df['supply_pct']
    df['legend'] = 'historical'
    df = df[['date', 'sector', 'demand_pct', 'supply_pct', 'gap_pct', 'legend']]
    # Use last quarter's gap to predict this quarter's
    # df = df.sort_values('date')
    # df['gap_lag1'] = df.groupby('sector')['gap_pct'].shift(1)
    # df['demand_lag1'] = df.groupby('sector')['demand_pct'].shift(1)
    
    return df

In [7]:
df = align_and_merge(df_supply, df_demand)
df = engineer_features(df)
df.head()

,date,sector,demand_pct,supply_pct,gap_pct,legend
0,2018-01-01,Agriculture,5.604762,1.414157,4.190605,historical
1,2018-01-01,Construction,15.530598,5.319443,10.211155,historical
2,2018-01-01,Manufacturing,26.170906,12.473264,13.697642,historical
3,2018-01-01,Mining and Quarrying,0.967193,1.051810,-0.084617,historical
4,2018-01-01,Services,51.726541,79.741325,-28.014785,historical


## Forecasting Models

### ARIMA: SARIMA

In [8]:
# Forecasting-only cell: build and use ARIMA (SARIMA) models on gap_pct per sector
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima

# Copy and prep data from already-engineered frame
history = df[['date', 'sector', 'gap_pct']].copy()

forecast_rows = []
for sector_name, group in history.groupby('sector'):
    g = group.sort_values('date').set_index('date')
    g = g.asfreq('QS-JAN')
    y = g['gap_pct']

    auto_model = auto_arima(
        y,
        seasonal=True,
        m=4,  # quarterly seasonality
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore',
        trace=True
    )

    # Fit SARIMAX using best params
    sarima_model = SARIMAX(
        y,
        order=auto_model.order,
        seasonal_order=auto_model.seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fit = sarima_model.fit(disp=False)

    future_dates = pd.date_range('2024-04-01', '2030-12-31', freq='QS-JAN')
    pred = fit.get_forecast(steps=len(future_dates))
    pred_mean = pred.predicted_mean

    sector_forecast = pd.DataFrame({
        'date': future_dates,
        'sector': sector_name,
        'demand_pct': np.nan,
        'supply_pct': np.nan,
        'gap_pct': pred_mean.values,
        'legend': 'forecast'
    })
    forecast_rows.append(sector_forecast)

forecast_gap_pct_arima = pd.concat(forecast_rows, ignore_index=True)

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[4] intercept   : AIC=-33.095, Time=0.45 sec
 ARIMA(0,1,0)(0,0,0)[4] intercept   : AIC=-38.483, Time=0.02 sec
 ARIMA(1,1,0)(1,0,0)[4] intercept   : AIC=-35.930, Time=0.09 sec
 ARIMA(0,1,1)(0,0,1)[4] intercept   : AIC=-35.929, Time=0.18 sec
 ARIMA(0,1,0)(0,0,0)[4]             : AIC=-40.136, Time=0.01 sec
 ARIMA(0,1,0)(1,0,0)[4] intercept   : AIC=-36.483, Time=0.04 sec
 ARIMA(0,1,0)(0,0,1)[4] intercept   : AIC=-36.483, Time=0.03 sec
 ARIMA(0,1,0)(1,0,1)[4] intercept   : AIC=-35.054, Time=0.14 sec
 ARIMA(1,1,0)(0,0,0)[4] intercept   : AIC=-37.544, Time=0.04 sec
 ARIMA(0,1,1)(0,0,0)[4] intercept   : AIC=-37.735, Time=0.04 sec
 ARIMA(1,1,1)(0,0,0)[4] intercept   : AIC=-35.735, Time=0.07 sec

Best model:  ARIMA(0,1,0)(0,0,0)[4]          
Total fit time: 1.127 seconds
Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,1,1)[4] intercept   : AIC=-10.673, Time=0.43 sec
 ARIMA(0,0,0)(0,1,0)[4] intercept   : AIC=-4.422, Time=0.

c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


 ARIMA(2,0,2)(1,0,1)[4] intercept   : AIC=8.802, Time=0.37 sec
 ARIMA(0,0,0)(0,0,0)[4] intercept   : AIC=46.945, Time=0.01 sec
 ARIMA(1,0,0)(1,0,0)[4] intercept   : AIC=7.738, Time=0.11 sec
 ARIMA(0,0,1)(0,0,1)[4] intercept   : AIC=inf, Time=0.13 sec
 ARIMA(0,0,0)(0,0,0)[4]             : AIC=203.023, Time=0.01 sec
 ARIMA(1,0,0)(0,0,0)[4] intercept   : AIC=7.198, Time=0.06 sec
 ARIMA(1,0,0)(0,0,1)[4] intercept   : AIC=7.641, Time=0.12 sec
 ARIMA(1,0,0)(1,0,1)[4] intercept   : AIC=inf, Time=0.28 sec
 ARIMA(2,0,0)(0,0,0)[4] intercept   : AIC=3.449, Time=0.05 sec
 ARIMA(2,0,0)(1,0,0)[4] intercept   : AIC=4.815, Time=0.20 sec
 ARIMA(2,0,0)(0,0,1)[4] intercept   : AIC=4.793, Time=0.20 sec
 ARIMA(2,0,0)(1,0,1)[4] intercept   : AIC=6.085, Time=0.29 sec
 ARIMA(3,0,0)(0,0,0)[4] intercept   : AIC=4.727, Time=0.13 sec
 ARIMA(2,0,1)(0,0,0)[4] intercept   : AIC=4.825, Time=0.19 sec
 ARIMA(1,0,1)(0,0,0)[4] intercept   : AIC=5.347, Time=0.10 sec
 ARIMA(3,0,1)(0,0,0)[4] intercept   : AIC=6.725, Time=0.

In [18]:
forecast_arima = forecast_gap_pct_arima
forecast_arima.head(10)

,date,sector,demand_pct,supply_pct,gap_pct,legend
0,2024-04-01,Agriculture,NaN,NaN,3.901461,forecast
1,2024-07-01,Agriculture,NaN,NaN,3.901461,forecast
2,2024-10-01,Agriculture,NaN,NaN,3.901461,forecast
3,2025-01-01,Agriculture,NaN,NaN,3.901461,forecast
4,2025-04-01,Agriculture,NaN,NaN,3.901461,forecast
5,2025-07-01,Agriculture,NaN,NaN,3.901461,forecast
6,2025-10-01,Agriculture,NaN,NaN,3.901461,forecast
7,2026-01-01,Agriculture,NaN,NaN,3.901461,forecast
8,2026-04-01,Agriculture,NaN,NaN,3.901461,forecast
9,2026-07-01,Agriculture,NaN,NaN,3.901461,forecast


### Holt Winters

In [10]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

forecast_rows = []

for sector_name, group in history.groupby('sector'):
    g = group.sort_values('date').set_index('date')
    y = g['gap_pct']

    model = ExponentialSmoothing(
        y,
        trend='add',
        seasonal='add',
        seasonal_periods=4
    ).fit()

    future_dates = pd.date_range('2024-03-01', '2030-12-31', freq='QS')
    forecast = model.forecast(len(future_dates))

    sector_forecast = pd.DataFrame({
        'date': future_dates,
        'sector': sector_name,
        'gap_pct_forecast_ets': forecast.values
    })

    forecast_rows.append(sector_forecast)

forecast_gap_pct_ets = pd.concat(forecast_rows, ignore_index=True)

c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QS-OCT will be used.
  self._init_dates(dates, freq)
c:\Users\Justin Lai\

In [19]:
forecast_ets = forecast_gap_pct_ets[forecast_gap_pct_ets['sector'] == 'Mining and Quarrying']
forecast_ets.head(10)

,date,sector,gap_pct_forecast_ets
81,2024-04-01,Mining and Quarrying,-0.081262
82,2024-07-01,Mining and Quarrying,-0.087913
83,2024-10-01,Mining and Quarrying,-0.086449
84,2025-01-01,Mining and Quarrying,-0.087221
85,2025-04-01,Mining and Quarrying,-0.081634
86,2025-07-01,Mining and Quarrying,-0.088285
87,2025-10-01,Mining and Quarrying,-0.086821
88,2026-01-01,Mining and Quarrying,-0.087593
89,2026-04-01,Mining and Quarrying,-0.082005
90,2026-07-01,Mining and Quarrying,-0.088656


## Evaluation

In [21]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pmdarima import auto_arima

comparison_results = []

for sector_name, group in history.groupby('sector'):
    g = group.sort_values('date').set_index('date')
    g = g.asfreq('QS-JAN')  # important alignment
    y = g['gap_pct']

    # =========================
    # 🔹 Train-Test Split
    # =========================
    train_size = int(len(y) * 0.8)
    y_train = y.iloc[:train_size]
    y_test = y.iloc[train_size:]

    # =========================
    # 🔹 SARIMA
    # =========================

    try:
        # =========================
        # 🔹 Auto ARIMA
        # =========================
        auto_model = auto_arima(
            y_train,
            seasonal=True,
            m=4,  # quarterly seasonality
            stepwise=True,
            suppress_warnings=True,
            error_action='ignore',
            trace=False
        )

        # Fit SARIMAX using best params
        sarima_model = SARIMAX(
            y_train,
            order=auto_model.order,
            seasonal_order=auto_model.seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        sarima_fit = sarima_model.fit(disp=False)

        sarima_pred = sarima_fit.get_forecast(
            steps=len(y_test)
        ).predicted_mean

    except:
        sarima_pred = pd.Series([np.nan]*len(y_test), index=y_test.index)

    # =========================
    # 🔹 Holt-Winters (ETS)
    # =========================
    try:
        hw_model = ExponentialSmoothing(
            y_train,
            trend='add',
            seasonal='add',
            seasonal_periods=4
        ).fit()
        hw_pred = hw_model.forecast(len(y_test))
    except:
        hw_pred = pd.Series([np.nan]*len(y_test), index=y_test.index)

    # =========================
    # 🔹 Metrics Function
    # =========================
    def get_metrics(y_true, y_pred):
        mask = ~np.isnan(y_pred)
        y_true = y_true[mask]
        y_pred = y_pred[mask]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        return mae, rmse, mape

    def get_metrics(y_true, y_pred):
        mask = ~np.isnan(y_pred)
        y_true = y_true[mask]
        y_pred = y_pred[mask]

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        return mae, rmse, mape
    hw_mae, hw_rmse, hw_mape = get_metrics(y_test, hw_pred)

    sarima_mae, sarima_rmse, sarima_mape = get_metrics(y_test, sarima_pred)

    comparison_results.append({
        'sector': sector_name,
        
        'SARIMA_RMSE': sarima_rmse,
        'HW_RMSE': hw_rmse,
        
        'SARIMA_MAE': sarima_mae,
        'HW_MAE': hw_mae,
        
        'SARIMA_MAPE': sarima_mape,
        'HW_MAPE': hw_mape,
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df['Best_Model_RMSE'] = np.where(
    comparison_df['SARIMA_RMSE'] < comparison_df['HW_RMSE'],
    'SARIMA',
    'Holt-Winters'
)

comparison_df

c:\Users\Justin Lai\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,sector,SARIMA_RMSE,HW_RMSE,SARIMA_MAE,HW_MAE,SARIMA_MAPE,HW_MAPE,Best_Model_RMSE
0,Agriculture,0.093719,0.161451,0.073094,0.131299,1.896698,3.412060,SARIMA
1,Construction,0.235362,0.427151,0.207527,0.399549,2.625921,5.061867,SARIMA
2,Manufacturing,0.093035,0.512031,0.076629,0.434689,0.541796,3.071146,SARIMA
3,Mining and Quarrying,0.038388,0.010705,0.035221,0.009041,43.131822,10.973114,Holt-Winters
4,Services,0.280465,0.509523,0.239727,0.439895,0.929194,1.705259,SARIMA


## Writing table to database

In [ ]:
df_final = pd.concat([df, forecast_gap_pct_arima], ignore_index=True)
df_final = df_final.sort_values(['date','sector']).reset_index(drop=True)
df_final.head(20)

write_table(df_final, 'sc_silver','forecast_supply_demand')

## XGBoost

In [14]:
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error
# import numpy as np

# metrics_list = []
# forecast_rows = []

# for sector_name, group in history.groupby('sector'):
#     g = group.sort_values('date').copy()

#     # Create lag features
#     for lag in range(1, 5):
#         g[f'lag_{lag}'] = g['gap_pct'].shift(lag)

#     g = g.dropna()

#     # =========================
#     # 🔹 Train-Test Split
#     # =========================
#     train_size = int(len(g) * 0.8)
#     train = g.iloc[:train_size]
#     test = g.iloc[train_size:]

#     X_train = train[[f'lag_{i}' for i in range(1, 5)]]
#     y_train = train['gap_pct']

#     X_test = test[[f'lag_{i}' for i in range(1, 5)]]
#     y_test = test['gap_pct']

#     # =========================
#     # 🔹 Train Model
#     # =========================
#     model = XGBRegressor()
#     model.fit(X_train, y_train)

#     # =========================
#     # 🔹 Predict on Test
#     # =========================
#     y_pred = model.predict(X_test)

#     # =========================
#     # 🔹 Metrics
#     # =========================
#     mae = mean_absolute_error(y_test, y_pred)
#     rmse = np.sqrt(mean_squared_error(y_test, y_pred))
#     mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
#     r2 = r2_score(y_test, y_pred)
#     metrics_list.append({
#         'sector': sector_name,
#         'MAE': mae,
#         'RMSE': rmse,
#         'MAPE (%)': mape
#     })

#     # =========================
#     # 🔹 Forecast Future
#     # =========================
#     future_dates = pd.date_range('2024-04-01', '2030-12-31', freq='QS')

#     last_vals = list(g['gap_pct'].tail(4))
#     preds = []

#     for _ in range(len(future_dates)):
#         X_input = np.array(last_vals[-4:]).reshape(1, -1)
#         pred = model.predict(X_input)[0]
#         preds.append(pred)
#         last_vals.append(pred)

#     sector_forecast = pd.DataFrame({
#         'date': future_dates,
#         'sector': sector_name,
#         'gap_pct_forecast_xgb': preds
#     })

#     forecast_rows.append(sector_forecast)

# # Combine outputs
# forecast_gap_pct_xgb = pd.concat(forecast_rows, ignore_index=True)
# metrics_df = pd.DataFrame(metrics_list)

# print("=== Performance Metrics by Sector ===")
# print(metrics_df)